# Kokoro JParaCrawl · S2UT Baseline

1. **CPU** で PREPARE_AUDIO_PACKS=True にして、QCリリースを読み込み、検証付きWAV ZIPをDriveへ作成します。
2. **GPU** に切り替え、同じ設定で PREPARE_AUDIO_PACKS=False にし、先頭から実行します。HuBERT L6/KM100 unit・辞書・fairseqデータを準備します。
3. RUN_BASELINE=True にして最後のセルを実行します。固定ベンチマークのFisherレシピで **400,000 updates** 学習します。切断後は同じ設定で上から再実行します。
4. Baseline完了後、[比較Notebook](benchmark_comparison.ipynb)で CORPUS_MODE='kokoro' と同じ実験名を指定します。

Kokoroリリースは読み取り専用です。WAVからの80-bin FBANK生成はnative fairseqが担当します。ZIP展開とチェックポイントに十分なVMディスク空きが必要です。

In [ ]:
#@title 共通ログ：開始・逐次出力・生存通知・完了・失敗
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/kokoro_baseline/bootstrap')


In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


In [ ]:
#@title 実験名と実行スイッチ
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '5f87769a618296b44bb3949fb3f39f3b8af819fa'
KOKORO = True

KOKORO_RUN_ID = 'kokoro-jparacrawl-l4-v1' #@param {type:"string"}
KOKORO_CORPUS_OVERRIDE = '' #@param {type:"string"}
if KOKORO:
    if Path(KOKORO_RUN_ID).name != KOKORO_RUN_ID or KOKORO_RUN_ID in ('', '.', '..'):
        raise ValueError('KOKORO_RUN_IDは単一の実験名にしてください')
    ROOT = str(Path('/content/alignment-transfer-work') / KOKORO_RUN_ID)
    PERSISTENT = str(Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / KOKORO_RUN_ID)

PREPARE_AUDIO_PACKS = True #@param {type:"boolean"}
RUN_BASELINE = False #@param {type:"boolean"}

## 1. 固定環境の構築

In [ ]:
import os
import sys
PYTHON = '/content/alignment-venv/bin/python'
REPO = Path('/content/alignment-transfer-s2ut')
LOGS = Path(ROOT) / 'execution'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
BENCHMARK_RUNTIME_REVISION = '4361e55a99c7f4ea3a00c5c138f6820c34a07d06'
os.environ.update(HF_HOME='/content/huggingface', MPLBACKEND='Agg', PYTHONUNBUFFERED='1', PYTHONUTF8='1',
                  S2ST_DRIVE_SAFE='1', S2ST_BACKUP_MIN_SECONDS='600', S2ST_BACKUP_MAX_GB='1024')

if KOKORO:
    DATA = Path(ROOT) / 'data'
    COMMON = DATA / 'common'
    BENCHMARK_UNITS = DATA / 's2ut/units'
    PREPARED = DATA / 's2ut/fairseq'
    BACKUP = Path(PERSISTENT) / 'baseline'
    RESTORED = Path(ROOT) / 'baseline'

In [ ]:
#@title Python・fairseq環境を構築
if not (REPO / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', REPO_URL, str(REPO), name='clone-alignment')
execute('git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION, name='fetch-alignment')
execute('git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION, name='checkout-alignment')
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)


In [ ]:
#@title benchmark runtimeを固定revisionで導入
if not (BENCHMARK_RUNTIME / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', str(BENCHMARK_RUNTIME))
execute('git', '-C', str(BENCHMARK_RUNTIME), 'fetch', 'origin', BENCHMARK_RUNTIME_REVISION, name='fetch-benchmark-runtime')
execute('git', '-C', str(BENCHMARK_RUNTIME), 'checkout', '--detach', BENCHMARK_RUNTIME_REVISION, name='checkout-benchmark-runtime')
execute(PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME))
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
if KOKORO:
    config = execute_json(PYTHON, '-c',
        "import json, yaml; print(json.dumps(yaml.safe_load(open('configs/data/colab.yaml', encoding='utf-8'))))",
        name='kokoro-paths')
    CORPUS_ROOT = Path(KOKORO_CORPUS_OVERRIDE or config['corpus_root'])
    SOURCE_MANIFEST = (CORPUS_ROOT / 'manifests/releases/accepted.jsonl'
                       if KOKORO_CORPUS_OVERRIDE else Path(config['accepted_manifest']))
    RELEASE_SUMMARY = SOURCE_MANIFEST.with_name('summary.json')
    if any(path.resolve().is_relative_to(CORPUS_ROOT.resolve()) for path in (Path(ROOT), Path(PERSISTENT))):
        raise ValueError('出力先をコーパス原本の外に指定してください')
    os.environ.pop('S2ST_INPUT_MAP', None)
    execute(PYTHON, '-m', 'pip', 'install', 'editdistance==0.8.1', 'librosa==0.10.2.post1',
        'accelerate==1.12.0', 'pandas==2.2.3', 'psutil==7.2.2', name='kokoro-runtime-dependencies')
    execute(PYTHON, '-m', 'pip', 'check', name='kokoro-dependency-check')

In [ ]:
execute(PYTHON, '-m', 'alignment_transfer_s2ut.environment', '--native', '--benchmark',
    *([] if PREPARE_AUDIO_PACKS else ['--cuda']), name='kokoro-environment')
if PREPARE_AUDIO_PACKS:
    log_event('kokoro-native-fp16-test', 'skipped', LOGS, 'CPU audio preparation')
else:
    execute(PYTHON, '-c',
        'from direct_s2st.s2ut.paper_recipe import preflight_dependencies; print(preflight_dependencies())',
        name='kokoro-paper-dependencies')
    execute(PYTHON, '-m', 'pytest', 'tests/test_benchmark_native.py::test_native_fp16_trainer_and_fp32_prior',
        'tests/test_kokoro.py::test_kokoro_release_to_native_recipe[cuda]',
        '-v', '-s', '--tb=short', '--basetemp', '/content/kokoro-native-fp16-test',
        cwd=REPO, name='kokoro-native-fp16-test')


## 2. QC合格manifestと音声ZIP

CPUで作った検証済みZIPをGPUランタイムで展開します。元のWAVはZIP作成時だけ読みます。

In [ ]:
def kokoro_stage(name, module, *arguments, work, restore_only=False):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(work),
        '--backup', str(Path(PERSISTENT) / 'stages' / name)]
    if restore_only:
        command.append('--restore-only')
    execute(*command, '--', PYTHON, '-m', module, *map(str, arguments), name=name)

def kokoro_common(restore_only=False):
    os.environ['CORPUS_ROOT'] = str(CORPUS_ROOT)
    kokoro_stage('kokoro-common', 'alignment_transfer_s2ut.benchmark_data', 'kokoro',
        '--source', SOURCE_MANIFEST, '--summary', RELEASE_SUMMARY, '--corpus-root', CORPUS_ROOT,
        '--output', COMMON, '--resume', work=COMMON, restore_only=restore_only)
    lock = json.loads((COMMON / 'dataset-lock.json').read_text(encoding='utf-8'))
    if lock['total_pairs'] != config['kokoro_expected_pairs']:
        raise ValueError(f"Kokoro train/dev count {lock['total_pairs']} differs from the configured release")
    print({'pairs': lock['total_pairs'], 'splits': lock['splits'],
        'estimated_waveform_GiB': round(lock['wav_bytes_estimate'] / 1024**3, 1)})

def kokoro_stage_audio(create=False):
    staged = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-kokoro',
        '--common', str(COMMON), '--corpus-root', str(CORPUS_ROOT), '--packs', str(Path(PERSISTENT) / 'wav-packs'),
        '--local', str(Path(ROOT) / 'staged-inputs'), *(['--create-audio'] if create else []), name='kokoro-audio')
    if not create:
        os.environ['S2ST_INPUT_MAP'] = staged['input_map']
    print(staged)

def kokoro_prepare(restore_only=False):
    kokoro_stage('kokoro-benchmark-units', 'alignment_transfer_s2ut.kokoro_runtime', 'units',
        '--common', COMMON, '--output', BENCHMARK_UNITS, '--cache', '/content/alignment-assets',
        work=BENCHMARK_UNITS, restore_only=restore_only)
    kokoro_stage('kokoro-fairseq', 'alignment_transfer_s2ut.kokoro_runtime', 'prepare',
        '--common', COMMON, '--units', BENCHMARK_UNITS, '--output', PREPARED,
        work=PREPARED, restore_only=restore_only)

def kokoro_baseline(restore_only=False):
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.kokoro_runtime', 'baseline',
        '--data', str(DATA), '--work', str(RESTORED), '--backup', str(BACKUP),
        *(['--restore-only'] if restore_only else ['--train']), name='kokoro-baseline')

In [ ]:
kokoro_common()
kokoro_stage_audio(create=PREPARE_AUDIO_PACKS)
if PREPARE_AUDIO_PACKS:
    print('WAV ZIPの準備完了。GPUに切り替え、PREPARE_AUDIO_PACKS=Falseで上から再実行してください。')

## 3. HuBERT unit・辞書・fairseq学習データ

完了した前処理は検証済みZIPから再開します。

In [ ]:
if PREPARE_AUDIO_PACKS:
    log_event('kokoro-s2ut-preparation', 'skipped', LOGS, 'CPU audio preparation')
else:
    kokoro_prepare()
    execute(PYTHON, '-c',
        'import sys; from alignment_transfer_s2ut.benchmark import prepared_identity; '
        'print(prepared_identity(sys.argv[1])["data_lock"]["splits"])',
        str(PREPARED), name='kokoro-registration-preflight')


## 4. Native S2UT Baseline

GPU接続試験とprepared dataの登録事前検査を確認してから、RUN_BASELINE=True にします。Driveに公開された最後の検証済みcheckpointから再開します。

In [ ]:
if PREPARE_AUDIO_PACKS:
    log_event('kokoro-baseline', 'skipped', LOGS, 'CPU audio preparation')
elif RUN_BASELINE:
    kokoro_baseline()
else:
    log_event('kokoro-baseline', 'skipped', LOGS, 'RUN_BASELINE=False')